# 🔬 MorphoBricks preprocessing with Cellpose

Prepares a microscopy experiment for the other MorphoBricks bricks, as the **MorphoBricks preprocessing** notebook does, then segments it with **Cellpose 4.2.1.1** and registers the masks, so the run is ready for MorphoFeature in one notebook. Your images are never changed: the histograms go to a separate output folder, the masks to a separate masks folder.

### What it does

1. **Preprocessing**, the same app as the MorphoBricks preprocessing notebook (fast mode included): **Images → Axes → Channels → Masks → Design → Check and save → Run**.
2. **Segment with Cellpose** (optional) the run the app just finished: the output (cell mask, nuclei mask or both), the channels **by channel ID**, the model and settings, a preview of one example, then the whole experiment. Each mask type gets its own folder with a `segmentation_record.json`.
3. **Register the masks:** confirm the declaration of each new mask type; the app then checks the folders again with the masks folder, saves the description, runs again (the histograms are reused) and describes the masks.

Nothing is guessed: what is proposed is recorded only when you click or tick.

### Before you start

- One experiment folder of TIFF (.tif/.tiff) or OME-Zarr images, with folders for the conditions and biological repeats.
- A masks folder for Cellpose's masks, **outside** the experiment and the output folder (it may already hold other masks).
- Cellpose segments **2D images only** (no Z-stacks or time-lapses) and reads at most three channels.
- **Google Colab:** keep the experiment, the output and the masks folders on Google Drive, under **/content/drive/MyDrive/**, and use a GPU runtime for Cellpose.
- **JupyterLab / LabConstrictor:** use normal local paths.

### How to run

Run the code cells in order (click a cell and press **Shift + Enter**, or its ▶ button):

1. **Install** prepares preprocessing (not Cellpose).
2. **Mount Google Drive** connects your Drive on Google Colab; elsewhere it does nothing.
3. **Run preprocessing** opens the app: run the experiment there.
4. **Segment with Cellpose** installs Cellpose (with PyTorch) the first time, only when you tick it, then segments the run of cell 3.
5. **Register the masks** declares the new masks, runs again and describes them.

Cells 1 to 3 are exactly those of the MorphoBricks preprocessing notebook. Cellpose, PyTorch and the model weights are downloaded only in cell 4, and only when you tick their boxes.

___

## **1. Install MorphoBricks preprocessing**

Installs the preprocessing commit this notebook is made for (it holds the segmentation and registration steps of cells 4 and 5), with its notebook dependencies (on Google Colab also the OME-Zarr reader), into the Python environment of this notebook. It does **not** install Cellpose or PyTorch: cell 4 does, when you ask. Nothing is installed when it is already there, so running the cell again is safe.

When it finishes, it prints **Ready** with the versions of preprocessing, normalization and ipywidgets, the notebook version and where the package was loaded from. No restart of the runtime is needed.

In [ ]:
# @title 1. Install MorphoBricks preprocessing
import platform
import subprocess
import sys


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback (Colab and Jupyter)."""

    def _render_traceback_(self) -> list:
        return [str(self)]


# Checked first, before anything that needs Python 3.11 (tomllib), so 3.10 gets this one line.
if sys.version_info < (3, 11):
    raise CellStop("Select a Python 3.11 or newer kernel, then run this cell again.")

import json
from importlib import import_module, invalidate_caches
from importlib.metadata import PackageNotFoundError, distribution, version
from importlib.util import find_spec
from pathlib import Path
from urllib.parse import urlsplit
from urllib.request import url2pathname

# LabConstrictor version identifiers.
current_version = "0.8.0"
notebook_name = "MorphoBricks_Preprocessing_Cellpose"

# A downloaded/Colab notebook installs this immutable source revision.
RELEASE_TAG = "v0.8.0"
RELEASE_REF = "3af789160e9ce224a60da4582e9371f028ff47c5"
REPOSITORY = "https://github.com/CellMigrationLab/morphobricks-preprocessing.git"
EXTRAS = "notebook"

# A local checkout (nearby, or an editable install) is used only when it is exactly the
# RELEASE_REF commit with no uncommitted changes; an installed package only when pip recorded
# that it installed RELEASE_REF from REPOSITORY. DEVELOPMENT_OVERRIDE covers local checkouts
# only: True uses a checkout of another commit, with uncommitted changes or of unknown state as
# it is, and the printed header and SETUP_RECORD then say so (the commit is shown either way).
# It never keeps an installed, non-editable package of another commit: that is still replaced
# by RELEASE_REF. Set True only to develop against such a checkout.
DEVELOPMENT_OVERRIDE = False
SETUP_RECORD: dict = {}

if "google.colab" in sys.modules:
    print("Google Colab: preparing MorphoBricks preprocessing...", flush=True)
    EXTRAS = "notebook,zarr"
else:
    print(
        f"JupyterLab: Python {platform.python_version()} on {platform.platform()}\n"
        f"Working folder: {Path.cwd()}",
        flush=True,
    )


def _direct_url(name: str) -> dict:
    """Return what pip recorded about where a package came from (PEP 610), or an empty dict.

    Empty when the package is not installed or its direct_url.json is missing or unreadable.
    """
    try:
        text = distribution(name).read_text("direct_url.json")
        found = json.loads(text) if text else {}
    except (PackageNotFoundError, OSError, ValueError):
        return {}
    return found if isinstance(found, dict) else {}


def _installed_commit(name: str) -> str | None:
    """Return the commit pip installed a package at from REPOSITORY with Git, else None.

    Only pip's record counts, never the version number: None for another URL, a local folder,
    an editable install or an archive, and for a missing or unreadable record.
    """
    found = _direct_url(name)
    info = found.get("vcs_info")
    if found.get("url") != REPOSITORY or not isinstance(info, dict) or info.get("vcs") != "git":
        return None
    commit = info.get("commit_id")
    return commit if isinstance(commit, str) else None


def _replace_other_commit(name: str, target: str) -> None:
    """Reinstall ``target`` over an installed ``name`` that pip did not record at RELEASE_REF.

    pip takes an installed package of the same version as already satisfied, even when asked
    for another commit, and installs nothing; so the pinned commit is forced. Only ``name`` is
    replaced: ``--no-deps`` never reinstalls numpy or another package the running kernel has
    loaded (the install after this one adds what is missing). pip restores the old copy if the
    new one cannot be installed. Nothing is done when ``name`` is not installed.
    """
    try:
        distribution(name)
    except PackageNotFoundError:
        return
    if _installed_commit(name) != RELEASE_REF:
        print(f"Replacing the installed {name} with commit {RELEASE_REF[:12]}...", flush=True)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--disable-pip-version-check",
                "--force-reinstall",
                "--no-deps",
                target,
            ],
            check=True,
        )


def _loaded(name: str) -> bool:
    """Return whether this kernel already imported the package that ``name`` installs.

    A module imported before pip replaced its files stays the old copy until the kernel
    restarts. The import name is the distribution name with ``_`` for ``-``.
    """
    return name.replace("-", "_") in sys.modules


def _require_installed_copy(name: str, module_file: str) -> None:
    """Stop when this kernel imported ``name`` from outside the folder pip installed it in.

    pip's record describes only the installed files. A folder earlier on ``sys.path``, such as
    a checkout's ``src`` folder used as the working folder, is imported instead while the record
    still names the pinned commit.
    """
    installed = Path(distribution(name).locate_file("")).resolve()
    if not Path(module_file).resolve().is_relative_to(installed):
        raise CellStop(
            f"This kernel imports {name} from {module_file}, not from the copy pip installed in "
            f"{installed}. Open the notebook from a folder without another copy, restart the "
            "kernel, then run this cell again."
        )


def _editable_checkout(name: str) -> Path | None:
    """Return the local folder an editable install of a package runs from (PEP 610), or None."""
    found = _direct_url(name)
    info, url = found.get("dir_info"), found.get("url")
    if not isinstance(info, dict) or info.get("editable") is not True or not isinstance(url, str):
        return None
    parts = urlsplit(url)
    return Path(url2pathname(parts.path)) if parts.scheme == "file" else None


def _checkout_version(checkout: Path) -> str | None:
    """Return the nearby checkout package version, or None when it cannot be read."""
    import tomllib  # Python 3.11 or newer, checked at the top of this cell

    try:
        data = tomllib.loads((checkout / "pyproject.toml").read_text(encoding="utf-8"))
    except (OSError, tomllib.TOMLDecodeError):
        return None
    value = data.get("project", {}).get("version")
    return str(value) if value is not None else None


def _git(checkout: Path, *arguments: str) -> str | None:
    """Return the output of ``git -C checkout ...``, or None when Git cannot answer."""
    try:
        return subprocess.check_output(
            ["git", "-C", str(checkout), *arguments], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (subprocess.CalledProcessError, OSError):
        return None


def _checkout_identity(checkout: Path) -> tuple[str | None, bool | None]:
    """Return the checkout's Git commit and whether its tree has changes (None when unknown).

    Unknown means not a Git work tree rooted at the checkout, or Git is not available: the code
    identity cannot then be shown, and it is never guessed.
    """
    top = _git(checkout, "rev-parse", "--show-toplevel")
    if top is None or Path(top).resolve() != checkout.resolve():
        return None, None
    commit = _git(checkout, "rev-parse", "HEAD")
    status = _git(checkout, "status", "--porcelain")
    return commit, (None if status is None else bool(status))


def _require_release_checkout(checkout: Path) -> tuple[str | None, bool | None]:
    """Show the checkout's commit and tree state; refuse one that is not the clean release."""
    commit, dirty = _checkout_identity(checkout)
    differences = []
    if commit != RELEASE_REF:
        differences.append(
            f"its commit is {commit or 'unknown (not a Git work tree, or Git is unavailable)'}, "
            f"not RELEASE_REF {RELEASE_REF}"
        )
    if dirty is not False:
        differences.append(
            "it has uncommitted changes" if dirty else "its tree state could not be read"
        )
    if differences and not DEVELOPMENT_OVERRIDE:
        raise CellStop(
            f"The local checkout {checkout} is not the release this notebook pins: "
            + "; ".join(differences)
            + f". Check out {RELEASE_REF} with a clean tree, or set DEVELOPMENT_OVERRIDE = True "
            "at the top of this cell to develop against this checkout, then run the cell again."
        )
    print(f"Local checkout {checkout}: {_describe_checkout(commit, dirty)}", flush=True)
    if DEVELOPMENT_OVERRIDE:
        print(
            "DEVELOPMENT OVERRIDE is on: this is not necessarily the release "
            f"{RELEASE_TAG} ({RELEASE_REF[:12]}).",
            flush=True,
        )
    return commit, dirty


def _record_setup(
    origin: str, package_version: str, checkout: Path | None, commit: str | None, dirty: bool | None
) -> str:
    """Record what this kernel loaded in SETUP_RECORD and return the header's checkout line."""
    SETUP_RECORD.clear()
    SETUP_RECORD.update(
        notebook_version=current_version,
        preprocessing_version=package_version,
        preprocessing_commit=commit,
        source=origin,
        release_ref=RELEASE_REF,
        checkout_commit=None if checkout is None else commit,
        checkout_has_uncommitted_changes=dirty,
        development_override=DEVELOPMENT_OVERRIDE,
    )
    if checkout is None:
        return ""
    override = "; DEVELOPMENT OVERRIDE on" if DEVELOPMENT_OVERRIDE else ""
    return f"Checkout: {_describe_checkout(commit, dirty)}{override}.\n"


def _describe_checkout(commit: str | None, dirty: bool | None) -> str:
    """Return the checkout's commit and tree state as one line."""
    state = {True: "uncommitted changes", False: "clean tree", None: "tree state unknown"}[dirty]
    return f"commit {commit or 'unknown'}, {state}"


def _missing_requirements() -> list[str]:
    """Return notebook requirements this kernel cannot import."""
    required = (
        "morphobricks_preprocessing",
        "morphobricks_normalization",
        "numpy",
        "tifffile",
        "ipywidgets",
        "matplotlib",
        *(("zarr",) if "zarr" in EXTRAS.split(",") else ()),
    )
    return [name for name in required if find_spec(name) is None]


def _install_requirements(missing: list[str], install_target: list[str], replace: bool) -> None:
    """Install ``install_target`` with this kernel's pip for the ``missing`` requirements.

    ``replace``: the installed preprocessing is not the RELEASE_REF commit (or not installed), so
    pip changes it; a copy this kernel already imported then needs a restart.
    """
    print(
        "Installing notebook requirements (" + ", ".join(dict.fromkeys(missing)) + ")...",
        flush=True,
    )
    try:
        if replace:
            _replace_other_commit("morphobricks-preprocessing", install_target[0])
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--disable-pip-version-check",
                *install_target,
            ],
            check=True,
        )
    except (subprocess.CalledProcessError, OSError) as exc:
        raise CellStop(
            "Notebook setup could not install its requirements. Check the installation "
            "output and internet access, then run this cell again."
        ) from exc
    invalidate_caches()
    if replace and _loaded("morphobricks-preprocessing"):
        raise CellStop(
            "pip replaced MorphoBricks preprocessing while this kernel already has another "
            "copy loaded. Restart the kernel, then run this cell again: it installs nothing "
            "more."
        )


def _install_preprocessing() -> None:
    """Prepare this kernel: install what is missing, check the interface loads, print versions.

    Running it again installs nothing that is already there.
    """
    current = Path.cwd().resolve()
    nearby = next(
        (
            folder
            for folder in (current, *current.parents)
            if (folder / "pyproject.toml").is_file()
            and (folder / "src/morphobricks_preprocessing/__init__.py").is_file()
        ),
        None,
    )
    checkout = (
        nearby if nearby is not None and _checkout_version(nearby) == current_version else None
    )
    if checkout is None:
        # An editable install runs a local checkout's files: the same rule as a nearby one. It
        # is refused, never replaced, when not the clean release: a developer set it up.
        checkout = _editable_checkout("morphobricks-preprocessing")

    commit, dirty = _require_release_checkout(checkout) if checkout is not None else (None, None)
    if checkout is not None:
        source = checkout / "src"
        loaded = sys.modules.get("morphobricks_preprocessing")
        if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(source):
            raise CellStop(
                "This kernel already loaded another MorphoBricks preprocessing copy. "
                "Restart the kernel, then run this cell again."
            )
        sys.path[:] = [str(source), *(entry for entry in sys.path if entry != str(source))]
        invalidate_caches()
        install_target = ["-e", f"{checkout}[{EXTRAS}]"]
        origin = f"local checkout {checkout}"
    else:
        install_target = [f"morphobricks-preprocessing[{EXTRAS}] @ git+{REPOSITORY}@{RELEASE_REF}"]
        # The same version built from another commit (a development install) is not the release:
        # only the commit pip recorded counts.
        commit = _installed_commit("morphobricks-preprocessing")
        origin = (
            "installed package"
            if commit == RELEASE_REF
            else f"pinned {RELEASE_TAG} candidate ({RELEASE_REF[:12]})"
        )

    missing = _missing_requirements()
    if checkout is None and commit != RELEASE_REF:
        missing.append(f"morphobricks-preprocessing {RELEASE_TAG} ({RELEASE_REF[:12]})")

    if missing:
        _install_requirements(missing, install_target, checkout is None and commit != RELEASE_REF)

    remaining = _missing_requirements()
    if remaining:
        raise CellStop(
            "Installation finished, but this kernel still cannot import "
            + ", ".join(remaining)
            + ". Restart the kernel and run this cell again."
        )
    if checkout is None:
        commit = _installed_commit("morphobricks-preprocessing")
        if commit != RELEASE_REF:
            # A restart would not help: pip kept the files (cell 1 forced the reinstall).
            raise CellStop(
                "Installation finished, but pip kept another MorphoBricks preprocessing: it "
                f"recorded {commit or 'no commit from ' + REPOSITORY}, not {RELEASE_TAG} "
                f"({RELEASE_REF}). Replace it with: {sys.executable} -m pip install "
                f'--force-reinstall --no-deps "{install_target[0]}", then run this cell again.'
            )

    try:
        import_module("morphobricks_preprocessing.notebook_ui")
    except ImportError as exc:
        # The cause stays on the one line: a stop must never hide what failed.
        cause = " ".join(f"{type(exc).__name__}: {exc}".split())
        raise CellStop(
            "MorphoBricks preprocessing was found, but its notebook interface could not load "
            f"({cause})."
        ) from exc

    package_version = version("morphobricks-preprocessing")
    normalization_version = version("morphobricks-normalization")
    widget_version = version("ipywidgets")
    package_file = Path(import_module("morphobricks_preprocessing").__file__).resolve()
    if checkout is None:
        _require_installed_copy("morphobricks-preprocessing", str(package_file))
    checkout_line = _record_setup(origin, package_version, checkout, commit, dirty)
    print(
        f"Ready: preprocessing {package_version} (commit {commit or 'unknown'}); "
        f"normalization {normalization_version}; ipywidgets {widget_version}.\n"
        f"Notebook {current_version}; source: {origin}.\n"
        f"{checkout_line}"
        f"Loaded from: {package_file}\n"
        "Next: run cell 2 (Mount Google Drive), then cell 3 (Run preprocessing).",
        flush=True,
    )


_install_preprocessing()

___

## **2. Mount Google Drive**

**Google Colab:** connects your Google Drive. Allow access in the window that opens; your Drive is then the folder **/content/drive/MyDrive**. Keep the experiment and the output folder there: everything else on a Colab runtime is wiped when it disconnects.

**JupyterLab / LabConstrictor:** there is no Drive to mount. The cell only prints a note, and you use normal local paths in the app.

In [ ]:
# @title 2. Mount Google Drive
import sys


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback (Colab and Jupyter)."""

    def _render_traceback_(self) -> list:
        return [str(self)]


if "google.colab" in sys.modules:
    from google.colab import drive

    print("Google Colab: connecting Google Drive...", flush=True)
    try:
        drive.mount("/content/drive")
    except Exception as exc:
        # The cause stays on the one line: a stop must never hide what failed.
        cause = " ".join(f"{type(exc).__name__}: {exc}".split())
        raise CellStop(
            f"Google Drive could not be connected ({cause}). Allow Drive access, then run "
            "this cell again."
        ) from exc
    print("Google Drive is connected: your files are in /content/drive/MyDrive.", flush=True)
else:
    print("Not Google Colab: there is no Google Drive to mount here, so nothing was done.")

___

## **3. Run preprocessing**

Opens the preprocessing app. Run cell 1 first: without it this cell stops with one line naming it. On Google Colab without Drive mounted (cell 2) the app still opens, and a line says that only files on this Colab runtime are visible.

The app has seven steps, one on screen at a time; move with **Next** or click a step in the strip at the top:

1. **Images**: give the experiment folder (masks and output are optional), click **Scan the folders**, look at the example image, then **Looks right: read all image headers**.
2. **Axes**: what each image dimension is, and the pixel size.
3. **Channels**: confirm the image type (separate stains or colour), name what you stained, then tag each folder's channels.
4. **Masks**: say what each mask type is (nothing to do without masks). After the run, come back here for **Describe the masks** if later bricks use the masks.
5. **Design**: conditions, biological repeats and comparisons in one grid; **Looks right: confirm the design**.
6. **Check and save**: read your experiment in words and save it.
7. **Run**: computes the histograms; the intensity checks and **Compare groups** are collapsed panels after the run.

What the app proposes is recorded only when you click. Every wait shows its progress, and a greyed-out button says beside it what it waits for.

In [ ]:
# @title 3. Run preprocessing
import importlib
import sys
from pathlib import Path


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback (Colab and Jupyter)."""

    def _render_traceback_(self) -> list:
        return [str(self)]


if not globals().get("SETUP_RECORD"):
    raise CellStop(
        "MorphoBricks preprocessing is not installed in this session yet: run cell 1 "
        "(Install MorphoBricks preprocessing) first, then run this cell again."
    )
if "google.colab" in sys.modules:
    if not Path("/content/drive/MyDrive").is_dir():
        print(
            "Google Drive is not mounted: only files on this Colab runtime are visible, "
            "and they are deleted when the runtime stops.",
            flush=True,
        )
    # Colab shows a long output in a frame that scrolls on its own; the page should scroll.
    from google.colab import output as colab_output

    colab_output.no_vertical_scroll()

try:
    interface = importlib.import_module("morphobricks_preprocessing.notebook_ui")
except ImportError as exc:
    # The cause stays on the one line: a stop must never hide what failed.
    cause = " ".join(f"{type(exc).__name__}: {exc}".split())
    raise CellStop(
        "MorphoBricks preprocessing was found, but its notebook interface could not load "
        f"({cause}). Run cell 1 again."
    ) from exc
notebook = interface.PreprocessingNotebook()

import ipywidgets as widgets
from IPython.display import display

app = widgets.VBox(
    [notebook.welcome(), notebook.steps(), notebook.results_step()],
    layout=widgets.Layout(width="99%"),
)
display(app)

___

## **4. Segment with Cellpose**

Segments the run that cell 3's app just finished, in the masks folder that run uses (the masks folder the app checked, else the one the run recorded). The step says which run and masks folder it took. With no finished run in the app (no run yet, or a run that stopped), it says so and names cell 3; to segment an earlier finished run instead, type its output folder and a masks folder under **Open the run**. A run whose images changed since it ran is refused, with the reason.

**The first time:** Cellpose 4.2.1.1 and PyTorch are not installed by cell 1. Tick **Install Cellpose 4.2.1.1 (with PyTorch) now** and press **Install Cellpose 4.2.1.1** (a download of a few gigabytes). If the installation changes `numpy` or `opencv` while the runtime already has them loaded, the step says to restart the runtime (Colab: Runtime > Restart session); then run cells 1, 3 and 4 again: nothing more is installed.

Then the same steps as the MorphoBricks Cellpose notebook, one under the other: **Open the run**; the **output** (Cell mask, Nuclei mask or Both) and the channels **by channel ID**; the model, the device and the settings, then **Load the model** (built-in weights are downloaded from **Hugging Face** only if you tick the download box; a GPU that is not available is refused, never replaced by the CPU); **Preview** one example (nothing is written); **Segment the whole experiment** (existing masks are never overwritten unless you tick it). Cells and nuclei are never paired by label number: MorphoRelate relates them later, by geometry.

In [ ]:
# @title 4. Segment with Cellpose

try:
    from morphobricks_preprocessing.notebook_stop import CellStop
except ImportError:
    raise ImportError("Run cell 1 first: it installs what this notebook needs.") from None

if "notebook" not in globals():
    raise CellStop("Run cell 3 (Run preprocessing) first: this cell segments the run of its app.")

from IPython.display import display

from morphobricks_preprocessing.notebook_preprocessing_cellpose import segmentation_step

segmentation = segmentation_step(notebook)
display(segmentation)

___

## **5. Register the masks**

For each mask type cell 4 wrote, the step shows the declaration it would add to the experiment description, for example:

```
[masks.Cells]
source_type = "Cells"
representation = "instances"
```

Nothing is declared until you tick each one and press **Register the masks**. Then the app of cell 3 does the rest with its own steps: it checks the folders again with the masks folder, answers each mask on **Masks**, saves the description (**Check and save**), runs again (the histograms of unchanged images are reused) and describes the masks (**Describe the masks**). After its folder check the app may ask you to confirm the axes again (cell 3, step **Axes**): the step says so; confirm them there and press **Register the masks** again.

It refuses, and changes nothing, when a mask type collides with a mask the description already declares, when the masks folder is inside the experiment (or the output folder), when the masks were written for another run or masks folder than the app's, or when the images changed since the run.

In [ ]:
# @title 5. Register the masks

try:
    from morphobricks_preprocessing.notebook_stop import CellStop
except ImportError:
    raise ImportError("Run cell 1 first: it installs what this notebook needs.") from None

if "notebook" not in globals():
    raise CellStop("Run cell 3 (Run preprocessing) first.")
if "segmentation" not in globals():
    raise CellStop("Run cell 4 (Segment with Cellpose) first.")

from IPython.display import display

from morphobricks_preprocessing.notebook_mask_register import register_step

registration = register_step(notebook, segmentation)
display(registration)

___

## **Versions and citation**

Cell 1 pins the MorphoBricks preprocessing commit this notebook runs; cell 4 installs **Cellpose 4.2.1.1** and every `segmentation_record.json` records the Cellpose version, the model, its weights file and SHA-256, the device and every setting. If you use Cellpose, cite it as its README asks: Pachitariu, Rariden & Stringer, *Cellpose-SAM: superhuman generalization for cellular segmentation* (bioRxiv, 2025); Stringer, Wang, Michaelos & Pachitariu, *Cellpose: a generalist algorithm for cellular segmentation* (Nature Methods, 2021); Pachitariu & Stringer, *Cellpose 2.0: how to train your own model* (Nature Methods, 2022).